In [ ]:
# Cell 1 - Clone repository
%cd /content
!if [ ! -d VDT_GD2 ]; then git clone https://github.com/Darkin72/VDT_GD2.git; fi
%cd /content/VDT_GD2
!git pull


In [ ]:
# Cell 2 - Paper training configuration
DEVICE = "cuda"
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
PATIENCE = 50
MULTI_SCALE = "64, 128, 256"
AUGMENTATION = "flip H/V, rotation 0/90/180/270, brightness/contrast +/-0.1"

CONFIG = {
    # Paper: synthetic datasets use 1000 epochs, batch 16 and cosine annealing.
    "SOTS_ITS": {"epochs": 1000, "batch": 16, "lr": 2e-4, "scheduler": "cosine"},
    "SOTS_OTS": {"epochs": 1000, "batch": 16, "lr": 1e-4, "scheduler": "cosine"},
    # I-HAZE/O-HAZY are real-world datasets; use the paper's real-world settings.
    "I_HAZE": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
    "O_HAZY": {"epochs": 500, "batch": 8, "lr": 2e-4, "scheduler": "step"},
}
for dataset, cfg in CONFIG.items():
    print(dataset, cfg)
print("device:", DEVICE, "| weight_decay:", WEIGHT_DECAY, "| min_lr:", MIN_LR)
print("multi-scale:", MULTI_SCALE, "| augmentation:", AUGMENTATION)

EVAL_MAX_SIDE = 1024  # Match Bench.ipynb-style bounded full-image evaluation


In [ ]:
# Cell 3 - Train with the paper settings
!pip -q install pillow matplotlib numpy
%cd /content/VDT_GD2

c = CONFIG["I_HAZE"]
!python -m solution.wavelet_dehaze.train "dataset/I-HAZE/train" --val-data "dataset/I-HAZE/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --step-size 100 --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_i_haze.pt --history history_i_haze.json

c = CONFIG["O_HAZY"]
!python -m solution.wavelet_dehaze.train "dataset/O-HAZY/train" --val-data "dataset/O-HAZY/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --step-size 100 --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_o_hazy.pt --history history_o_hazy.json

SOTS_ROOT = "dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]"

c = CONFIG["SOTS_ITS"]
!python -m solution.wavelet_dehaze.train "{SOTS_ROOT}/indoor/train" --val-data "{SOTS_ROOT}/indoor/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_sots_its.pt --history history_sots_its.json

c = CONFIG["SOTS_OTS"]
!python -m solution.wavelet_dehaze.train "{SOTS_ROOT}/outdoor/train" --val-data "{SOTS_ROOT}/outdoor/val" --device {DEVICE} --epochs {c["epochs"]} --batch-size {c["batch"]} --size 256 --lr {c["lr"]} --weight-decay {WEIGHT_DECAY} --scheduler {c["scheduler"]} --min-lr {MIN_LR} --augment --multi-scale --patience {PATIENCE} --out haze_wavelet_sots_ots.pt --history history_sots_ots.json


In [ ]:
# Cell 4 - Preview metrics separately for each dataset
import json
from pathlib import Path
import matplotlib.pyplot as plt

files = {
    "I-HAZE": "history_i_haze.json",
    "O-HAZY": "history_o_hazy.json",
    "SOTS-ITS": "history_sots_its.json",
    "SOTS-OTS": "history_sots_ots.json",
}
histories = {
    name: json.loads(Path(path).read_text())
    for name, path in files.items()
    if Path(path).exists()
}

if not histories:
    raise FileNotFoundError("No history_*.json files found. Run Cell 3 first.")

fig, axes = plt.subplots(
    len(histories), 3,
    figsize=(20, 5 * len(histories)),
    squeeze=False,
)
for row, (name, records) in enumerate(histories.items()):
    epoch = [item["epoch"] for item in records]
    metrics = [
        ("train_loss", "val_loss", "Loss"),
        ("train_psnr", "val_psnr", "PSNR (dB)"),
        ("train_ssim", "val_ssim", "SSIM"),
    ]
    for col, (train_key, val_key, title) in enumerate(metrics):
        ax = axes[row, col]
        ax.plot(epoch, [item[train_key] for item in records], label="Train", linewidth=2)
        ax.plot(epoch, [item[val_key] for item in records], "--", label="Validation", linewidth=2)
        ax.set_title(f"{name} - {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(alpha=0.25)
        ax.legend()

plt.tight_layout()
plt.savefig("haze_wavelet_training_metrics_by_dataset.png", dpi=160)
plt.show()

In [ ]:
# Cell 5 - Evaluate four test sets and export four Excel workbooks
!pip -q install openpyxl psutil scikit-image
%cd /content/VDT_GD2

RESULT_DIR = "utils/evaluation_results/hazewavenet"
!mkdir -p {RESULT_DIR}

!python -m solution.wavelet_dehaze.evaluate --dataset "dataset/I-HAZE/test" --checkpoint haze_wavelet_i_haze.pt --device {DEVICE} --max-side {EVAL_MAX_SIDE} --name "I-HAZE" --output "{RESULT_DIR}/hazewavenet_i_haze_test.xlsx"

!python -m solution.wavelet_dehaze.evaluate --dataset "dataset/O-HAZY/test" --checkpoint haze_wavelet_o_hazy.pt --device {DEVICE} --max-side {EVAL_MAX_SIDE} --name "O-HAZY" --output "{RESULT_DIR}/hazewavenet_o_hazy_test.xlsx"

SOTS_ROOT = "dataset/Synthetic Objective Testing Set (SOTS) [RESIDE]"
!python -m solution.wavelet_dehaze.evaluate --dataset "{SOTS_ROOT}/indoor/test" --checkpoint haze_wavelet_sots_its.pt --device {DEVICE} --max-side {EVAL_MAX_SIDE} --name "SOTS-ITS" --output "{RESULT_DIR}/hazewavenet_sots_its_test.xlsx"

!python -m solution.wavelet_dehaze.evaluate --dataset "{SOTS_ROOT}/outdoor/test" --checkpoint haze_wavelet_sots_ots.pt --device {DEVICE} --max-side {EVAL_MAX_SIDE} --name "SOTS-OTS" --output "{RESULT_DIR}/hazewavenet_sots_ots_test.xlsx"

print("Excel files:")
for path in sorted(Path(RESULT_DIR).glob("*.xlsx")):
    print(" -", path)
